In [1]:
%reset -f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, gc, time, zipfile
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
# 将上一级目录添加到搜索路径中
import sys
sys.path.append(os.path.abspath(".."))
from evaluator import evaluate_all  # 确保导入了你上传的评估文件

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, accuracy_score, brier_score_loss, recall_score, precision_score)
from scipy import stats
import numpy as np

# 预测模型
from sklearn.linear_model import LogisticRegression    # Logit模型
from sklearn.ensemble import RandomForestClassifier     # 随机森林
from xgboost import XGBClassifier                       # XGBoost

# 评估与预处理
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler        # Logit 必须进行标准化
from sklearn.metrics import accuracy_score
from evaluator import evaluate_all

In [2]:
import joblib
file_path = "/root/autodl-tmp/DATA/data_bundles.pkl"
data_bundles = joblib.load(file_path)

In [3]:
metric_names = [
    "ROC-AUC","F1", "Recall@1%", "Recall@5%" ]

def fit_predict_model(model, X_train, y_train, X_test, threshold=0.5):
    """
    统一封装：训练 + 预测概率 + 生成预测标签, 只需要替换 model 的构造即可
    """
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(np.int8)

    # 某些模型（如logit）有 n_iter_，树模型没有
    n_iter = getattr(model, "n_iter_", None)
    if isinstance(n_iter, (list, np.ndarray)):
        n_iter = n_iter[0]
    return y_prob, y_pred, n_iter

# 函数
def undersample_indices(y, neg_pos_ratio=1, seed=42):
    """对训练集进行随机欠采样（Random Undersampling）"""

    # 确保标签为整数型 0/1
    y = np.asarray(y).astype(int)

    # 分别获取正样本和负样本的索引位置
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    # 正样本数量
    n_pos = len(pos_idx)
    if n_pos == 0:
        raise ValueError("训练集中没有正样本，无法进行欠采样。")
    n_neg_need = min(len(neg_idx), n_pos * int(neg_pos_ratio))

    # 在负样本中随机无放回抽取所需数量
    rng = np.random.default_rng(seed)
    neg_sel = rng.choice(neg_idx, size=n_neg_need, replace=False)

    # 合并正样本与抽取的负样本索引
    sel = np.concatenate([pos_idx, neg_sel])
    # 打乱顺序，避免样本按类别排序
    rng.shuffle(sel)
    return sel


# t 区间 CI（重复10次，df=9）
def t_ci(values, ci=0.95):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    n = len(v)
    if n < 2:
        return (np.nan, np.nan, np.nan)
    mean = v.mean()
    se = v.std(ddof=1) / np.sqrt(n)
    alpha = 1 - ci
    tcrit = stats.t.ppf(1 - alpha/2, df=n-1)
    lo = mean - tcrit * se
    hi = mean + tcrit * se
    return mean, lo, hi

def fmt_ci(mean, lo, hi, nd=4):
    if np.isnan(mean):
        return "NA"
    return f"{mean:.{nd}f}[{lo:.{nd}f}; {hi:.{nd}f}]"

In [4]:
all_data = data_bundles

In [5]:
main_cols = ['insider_trading', 'Stkcd', 'Trddt']
features = [col for col in all_data['set1']['train'].columns if col not in main_cols]

In [6]:
# 参数
target = "insider_trading"
sets = ["set1", "set2", "set3"]
neg_pos_ratios = [1,2,5]   # 负:正 = 1:1, 2:1, 5:1（可自行加，比如 10, 20）
n_rep = 10                   # 重复次数（df=9）
ci_level = 0.95
base_seed = 42
USE_CLASS_WEIGHT_BALANCED = False     # 欠采样时建议 False

In [7]:
for set_name in sets:
    train_df = data_bundles[set_name]["train"]
    test_df  = data_bundles[set_name]["test"]
    print(set_name, train_df.shape, test_df.shape)

set1 (3028146, 713) (1046498, 713)
set2 (3028146, 713) (1046498, 713)
set3 (3028146, 713) (1046498, 713)


In [8]:
# 并行过程
from joblib import Parallel, delayed

PARALLEL_JOBS = 20  # 建议先从 4~8 试试

def one_run_logit(seed, ratio, X_train_full, y_train_full, X_test, y_test):
    """
    单次 run：欠采样 -> 训练 -> 预测 -> 评估
    返回：met(指标dict), n_iter
    """
    # ① 欠采样（只作用训练集）
    sel = undersample_indices(y_train_full, neg_pos_ratio=ratio, seed=seed)
    X_train = X_train_full[sel]
    y_train = y_train_full[sel]

    # ② 构造模型（外层并行时，内层 n_jobs 必须=1）
    model = LogisticRegression(
        class_weight=("balanced" if USE_CLASS_WEIGHT_BALANCED else None),
        solver="saga",
        penalty="l2",
        C=0.5,
        n_jobs=1,            # 关键：并行时改成1，避免嵌套并行
        max_iter=1000,
        tol=1e-3,
        random_state=seed,
        verbose=0
    )

    # ③ 训练 + 预测（测试集固定）
    y_prob, y_pred, n_iter = fit_predict_model(model, X_train, y_train, X_test, threshold=0.5)

    # ④ 评估
    metrics_dict, _ = evaluate_all(y_test, y_prob, base_thr=0.5)
    
    met = {
        "ROC-AUC": round(metrics_dict["ROC-AUC"], 4),
        "F1": round(metrics_dict["F1"], 4),
        "Recall@1%": round(metrics_dict.get("Recall@1.00%", 0), 4),
        "Recall@5%": round(metrics_dict.get("Recall@5.00%", 0), 4)
    }
    return met, n_iter

In [ ]:
# 主过程：每个set输出一张表 
# 在固定的时间切分下，对每一套数据（set1/2/3），在不同欠采样比例下，重复多次随机训练 Logit 模型，在同一测试集上评估性能，
# 并用重复结果构造 95% 置信区间。

tables = {}  
for set_name in sets:
    print(f"\n==================== {set_name} ====================")

    train_df = data_bundles[set_name]["train"]
    test_df  = data_bundles[set_name]["test"]

    # 数据准备：一次性转 numpy
    X_train_full = train_df[features].to_numpy(dtype=np.float32, copy=False)
    y_train_full = train_df[target].to_numpy(dtype=np.int8, copy=False)

    X_test = test_df[features].to_numpy(dtype=np.float32, copy=False)
    y_test = test_df[target].to_numpy(dtype=np.int8, copy=False)

    print("train shape:", X_train_full.shape, "pos_rate:", float(y_train_full.mean()))
    print("test  shape:", X_test.shape, "pos_rate:", float(y_test.mean()))

    rows = []

    for ratio in neg_pos_ratios:
        print(f"\n--- 欠采样 负:正={ratio}:1 | 并行重复 {n_rep} 次 ---")

        # 初始化 store
        store = {k: [] for k in metric_names}

        # seeds
        seeds = [base_seed + r for r in range(n_rep)]

        # 并行跑 n_rep 次
        results = Parallel(
            n_jobs=PARALLEL_JOBS,
            backend="loky",
            verbose=0
        )(
            delayed(one_run_logit)(
                seed, ratio,
                X_train_full, y_train_full,
                X_test, y_test
            )
            for seed in seeds
        )
                
        # 汇总每次 run 的指标并打印核心关注点
        for i, (met, n_iter) in enumerate(results, start=1):
            # 1. 填充 store 字典用于后续计算均值
            for k in metric_names:
                store[k].append(met.get(k, np.nan))

            # 2. 提取关注的四个指标用于控制台实时显示
            auc_val = met.get("ROC-AUC", np.nan)
            f1_val  = met.get("F1", np.nan)
            r1_val  = met.get("Recall@1%", np.nan)
            r5_val  = met.get("Recall@5%", np.nan)
            
            metrics_str = (f"AUC={auc_val:.4f} | F1={f1_val:.4f} | "
                           f"R@1%={r1_val:.4f} | R@5%={r5_val:.4f}")

            if n_iter is None:
                print(f"  run {i:02d}/{n_rep} | {metrics_str}")
            else:
                it = n_iter[0] if isinstance(n_iter, (list, np.ndarray)) else n_iter
                print(f"  run {i:02d}/{n_rep} | n_iter={it} | {metrics_str}")        
                

        # 只输出均值，不输出95%CI
        row = {"Train undersample (neg:pos)": f"{ratio}:1"}
        for k in metric_names:
            row[k] = round(np.nanmean(store[k]), 4)
        rows.append(row)

    table = pd.DataFrame(rows)
    tables[set_name] = table

    print(f"\n>>> {set_name} 汇总表：")
    display(table)


==================== set1 ====================
train shape: (3028146, 710) pos_rate: 0.0014163782063348332
test  shape: (1046498, 710) pos_rate: 0.00017391337584973885

--- 欠采样 负:正=1:1 | 并行重复 10 次 ---


In [ ]:
SAVE_EXCEL = True
excel_path = "/root/autodl-tmp/Table4/xgboost_tables.xlsx"

# 自动创建路径中不存在的文件夹，防止路径错误
import os
os.makedirs(os.path.dirname(excel_path), exist_ok=True)

In [21]:
# 保存到Excel（每个set一个sheet）
if SAVE_EXCEL:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for set_name, df_out in tables.items():
            df_out.to_excel(writer, sheet_name=set_name, index=False)
    print("\n已保存：", excel_path)


已保存： /root/autodl-tmp/Table3/logit_tables.xlsx
